# BESTEST LBNL-style report — Case 900

Native ISO is the formal ASHRAE candidate. Modelica-resolved solar is a controlled-forcing comparison only.

In [ ]:
from pathlib import Path
import subprocess, sys, pandas as pd, matplotlib.pyplot as plt
ISO_MODE = 'modelica_solar'  # 'modelica_solar' or 'native'
cwd=Path.cwd().resolve(); root=next((p for p in (cwd,*cwd.parents) if (p/'scripts/run_case900_validation.py').is_file()), cwd/'2__validation/_BESTEST')
sys.path.insert(0,str(root/'scripts'))
from bestest_reporting import MODELICA_LABEL, annual_energy_table, daily_slice, peak_table, selected_hourly, selected_iso_mode
subprocess.run([sys.executable,'scripts/run_case900_validation.py'],cwd=root,check=True)
case='900'; out=root/'results/case900'; reference=root/'_ref/BESTEST_LBNL_all_cases_reference.md'; metrics=pd.read_csv(out/'case900_metrics.csv'); hourly=pd.read_csv(out/'case900_feb1_load_profile.csv'); iso_run_mode, ISO_LABEL=selected_iso_mode(ISO_MODE)
def range_plot(table,title):
    row=table.iloc[0]; values=row.drop(labels=['Case','ASHRAE lower','ASHRAE upper','Python range status']).astype(float); fig,ax=plt.subplots(figsize=(9,3.4)); ax.bar(values.index,values.values,color='#4C78A8'); ax.axhline(row['ASHRAE lower'],color='#555',linestyle='--',label='ASHRAE lower'); ax.axhline(row['ASHRAE upper'],color='#555',linestyle='--',label='ASHRAE upper'); ax.set(ylim=(0,1.2*max(values.max(),row['ASHRAE upper'])),ylabel='MWh',title=title); ax.legend(); ax.grid(axis='y',alpha=.2); plt.xticks(rotation=45,ha='right'); fig.tight_layout(); plt.show()

## Inputs and implementation mapping

Case 900 uses the heavyweight BESTEST construction; all other prescribed Case 600-common inputs are retained.

In [ ]:
import json
display(pd.DataFrame([json.loads((root/'inputs/case900.json').read_text())['constructions']]))
display(pd.read_csv(root/'inputs/case900_adapter_mapping.csv'))

## 1. Annual heating energy

Annual heating energy, MWh.

In [ ]:
heating=annual_energy_table(metrics,reference,case,'annual_heating_energy',ISO_MODE); display(heating); range_plot(heating,'Case 900: annual heating energy')

## 2. Annual cooling energy

Annual cooling energy, MWh. The published Modelica value is retained even though it is below the published lower limit.

In [ ]:
cooling=annual_energy_table(metrics,reference,case,'annual_cooling_energy',ISO_MODE); display(cooling); range_plot(cooling,'Case 900: annual cooling energy')

## 3–4. Peak heating and cooling load

Peak values and completed-hour occurrence times, kW.

In [ ]:
display(peak_table(metrics,reference,case,'peak_heating_load',ISO_MODE)); display(peak_table(metrics,reference,case,'peak_cooling_load',ISO_MODE))

## 5. Daily load profile

Case 900 heating and cooling hourly averages on 1 February. Completed hour 1 = 00:00–01:00 and 24 = 23:00–24:00; neither series is shifted.

In [ ]:
profile=daily_slice(selected_hourly(hourly,ISO_MODE),2,1); profile['run']=profile.implementation.map({'modelica':MODELICA_LABEL,'iso13790':ISO_LABEL}); fig,axes=plt.subplots(2,1,figsize=(8,5),sharex=True)
for run,frame in profile.groupby('run'):
    frame=frame.sort_values('hour'); axes[0].plot(frame.hour,frame.heating_load_W/1000,marker='o',markersize=3,linewidth=1.2,label=run); axes[1].plot(frame.hour,frame.cooling_load_W/1000,marker='o',markersize=3,linewidth=1.2,label=run)
axes[0].set(ylabel='Heating (kW)',title='Case 900: 1 February load profile'); axes[1].set(xlabel='Hour of Day',ylabel='Cooling (kW)',xlim=(1,24),xticks=range(1,25))
for ax in axes: ax.grid(alpha=.25); ax.legend(loc='upper center',ncol=2,fontsize=8,frameon=False)
fig.tight_layout(); plt.show()

## 6. Solar-forcing comparison

Annual zone solar gains, MWh.

In [ ]:
display(pd.read_csv(out/'case900_solar_comparison.csv'))

## 7. Controlled-forcing agreement

ISO with Modelica-resolved solar minus Modelica native; diagnostic only.

In [ ]:
display(pd.read_csv(out/'case900_controlled_forcing_comparison.csv'))

## 8. Case 600 / 900 comparison

Native annual ISO error, controlled-forcing residual, and native/modelica solar ratio.

In [ ]:
m600=pd.read_csv(root/'results/case600/case600_metrics.csv'); m900=metrics
def comparison_row(case_label,m):
    p=m.pivot(index='metric',columns=['implementation','run_mode'],values='value')
    if case_label=='600':
        solar=pd.read_csv(root/'results/case600_debug/solar_summary.csv').set_index('quantity'); ratio=solar.loc['total_zone_solar_gain_W','annual_MWh_or_MWh_m2']/solar.loc['modelica_total_zone_solar_gain_W','annual_MWh_or_MWh_m2']
    else:
        solar=pd.read_csv(root/'results/case900/case900_solar_comparison.csv').set_index('quantity'); ratio=solar.loc['native_rclib_zone_solar_MWh','value']/solar.loc['modelica_resolved_zone_solar_MWh','value']
    return {'case':case_label,'native ISO heating error vs Modelica (MWh)':p.loc['annual_heating_energy',('iso13790','native')]-p.loc['annual_heating_energy',('modelica','native')],'native ISO cooling error vs Modelica (MWh)':p.loc['annual_cooling_energy',('iso13790','native')]-p.loc['annual_cooling_energy',('modelica','native')],'controlled heating residual (MWh)':p.loc['annual_heating_energy',('iso13790','diagnostic_modelica_solar')]-p.loc['annual_heating_energy',('modelica','native')],'controlled cooling residual (MWh)':p.loc['annual_cooling_energy',('iso13790','diagnostic_modelica_solar')]-p.loc['annual_cooling_energy',('modelica','native')],'native/modelica solar ratio':ratio}
display(pd.DataFrame([comparison_row('600',m600),comparison_row('900',m900)]))

## Summary

Case 900 uses the same architecture as Case 600; native and controlled-forcing outputs remain separate.

In [ ]:
display(metrics[metrics.metric.isin(['annual_heating_energy','annual_cooling_energy'])].pivot(index=['implementation','run_mode'],columns='metric',values='value'))
print('READY FOR BATCH GENERALISATION — no new mapping, schema, or timestamp issue was exposed by Case 900.')